# Blending vs. logistic stacking — s6e8 OOF library

Two independent submissions built from the same OOF/test library:
1. **Rank blending** — greedy forward ensemble selection on rank-transformed predictions.
2. **Logistic stacking** — logistic regression meta-model on raw probabilities, with a small `C` search.

Both are evaluated on the same fixed 5-fold split so their scores are directly comparable.
A wall-clock budget (8h to keep searching, 9h hard ceiling) guards the search loops.

In [ ]:
import time
import numpy as np, pandas as pd, glob, os
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

START_TIME = time.time()
WRAP_UP_HOURS = 8   # stop searching past this point and finalize with whatever is best so far
TOTAL_BUDGET_HOURS = 9

def elapsed_hours():
    return (time.time() - START_TIME) / 3600

def time_is_up():
    return elapsed_hours() > WRAP_UP_HOURS

print(f"time budget: search until {WRAP_UP_HOURS}h, hard ceiling {TOTAL_BUDGET_HOURS}h")

yo


In [ ]:
BASE = "/kaggle/input/datasets/szymonkapiski/s6e8-oof-library-47-models"
L = f"{BASE}/oof"

keys      = pd.read_parquet(f"{BASE}/train_keys.parquet")
test_keys = pd.read_parquet(f"{BASE}/test_keys.parquet")
manifest  = pd.read_csv(f"{BASE}/manifest.csv")
y = keys["addicted_label"].values

names = sorted(os.path.basename(p)[4:-4] for p in glob.glob(f"{L}/oof_*.npy"))
O = np.column_stack([np.load(f"{L}/oof_{n}.npy") for n in names]).astype("float64")
T = np.column_stack([np.load(f"{L}/test_{n}.npy") for n in names]).astype("float64")

assert O.shape[0] == len(y) and T.shape[0] == len(test_keys), "row count mismatch vs keys files"
print(f"{len(names)} models loaded — OOF {O.shape}, test {T.shape}")

In [ ]:
def to_rank(a):
    """Column-wise rank in (0,1] — removes cross-model calibration differences, used for blending only."""
    return np.column_stack([rankdata(a[:, i]) / len(a) for i in range(a.shape[1])])

R_oof, R_test = to_rank(O), to_rank(T)

# one fixed split reused by both parts so their OOF AUCs are directly comparable
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(O, y))

## Part 1 — Rank blending

Greedy forward ensemble selection (Caruana-style, with replacement) on rank-transformed
OOF predictions. The only "fitted" thing is which models get picked, so it's evaluated
honestly by re-running the selection separately inside each training fold.

In [ ]:
def greedy_select(ranks, labels, max_picks=25, patience=3):
    """Forward-select models (with replacement) to maximize AUC of their running average."""
    n_models = ranks.shape[1]
    picks, running, best_score, stale = [], np.zeros(len(labels)), -1.0, 0
    for _ in range(max_picks):
        scores = [roc_auc_score(labels, (running * len(picks) + ranks[:, j]) / (len(picks) + 1))
                  for j in range(n_models)]
        j = int(np.argmax(scores))
        if scores[j] <= best_score + 1e-6:
            stale += 1
            if stale >= patience:
                break
        else:
            stale = 0
        picks.append(j)
        running = (running * (len(picks) - 1) + ranks[:, j]) / len(picks)
        best_score = scores[j]
    return picks, best_score

blend_oof = np.full(len(y), np.nan)
for itr, iva in FOLDS:
    if time_is_up():
        print("time budget reached mid blend-search, stopping early")
        break
    picks, _ = greedy_select(R_oof[itr], y[itr])
    blend_oof[iva] = R_oof[iva][:, picks].mean(axis=1)

done = ~np.isnan(blend_oof)
blend_auc = roc_auc_score(y[done], blend_oof[done])
print(f"blend OOF AUC {blend_auc:.5f} on {done.sum()}/{len(y)} rows  ({elapsed_hours():.2f}h elapsed)")

In [ ]:
# in-sample score below (fit and scored on all data) — blend_auc above is the real generalization estimate
final_picks, final_score = greedy_select(R_oof, y)
print(f"final blend picks {len(final_picks)} models, in-sample AUC {final_score:.5f}")

blend_test_pred = R_test[:, final_picks].mean(axis=1)
sub_blend = test_keys[["id"]].copy()
sub_blend["addicted_label"] = blend_test_pred
sub_blend.to_csv("submission_blend.csv", index=False)
sub_blend.head()

## Part 2 — Logistic stacking (raw probabilities)

Logistic regression meta-model on the raw OOF probabilities, searching a wide `C` grid
across L1/L2 penalties and a few random seeds (more candidates, not seed-averaging),
evaluated with the same fixed folds used for the blend above.

In [ ]:
C_GRID = list(np.logspace(-4, 2, 10))
PENALTY_GRID = ["l1", "l2"]
SEED_GRID = [0, 1, 2]   # extra candidates to search over, not averaged into the final model

def cv_auc_for(C, penalty, seed):
    oof = np.full(len(y), np.nan)
    for itr, iva in FOLDS:
        if time_is_up():
            return None, oof
        m = LogisticRegression(C=C, penalty=penalty, solver="liblinear", random_state=seed,
                                max_iter=3000).fit(O[itr], y[itr])
        oof[iva] = m.predict_proba(O[iva])[:, 1]
    done = ~np.isnan(oof)
    return roc_auc_score(y[done], oof[done]), oof

best_C, best_penalty, best_seed, best_auc, best_oof = None, None, None, -1.0, None
stop = False
for penalty in PENALTY_GRID:
    for C in C_GRID:
        for seed in SEED_GRID:
            if time_is_up():
                print("time budget reached mid logistic-search, stopping early")
                stop = True
                break
            auc, oof = cv_auc_for(C, penalty, seed)
            if auc is None:
                stop = True
                break
            if auc > best_auc:
                best_C, best_penalty, best_seed, best_auc, best_oof = C, penalty, seed, auc, oof
                print(f"new best: C={C:.5g}  penalty={penalty}  seed={seed}  "
                      f"OOF AUC {auc:.5f}  ({elapsed_hours():.2f}h elapsed)")
        if stop:
            break
    if stop:
        break

logistic_auc, logistic_oof = best_auc, best_oof
print(f"\nbest C={best_C:.5g}  penalty={best_penalty}  seed={best_seed}  "
      f"logistic OOF AUC {logistic_auc:.5f}")

In [ ]:
final_logit = LogisticRegression(C=best_C, penalty=best_penalty, solver="liblinear",
                                  random_state=best_seed, max_iter=3000).fit(O, y)
logistic_test_pred = final_logit.predict_proba(T)[:, 1]

sub_logistic = test_keys[["id"]].copy()
sub_logistic["addicted_label"] = logistic_test_pred
sub_logistic.to_csv("submission_logistic.csv", index=False)
sub_logistic.head()

In [ ]:
print(f"blend    OOF AUC = {blend_auc:.5f}")
print(f"logistic OOF AUC = {logistic_auc:.5f}")

chosen = "logistic" if logistic_auc >= blend_auc else "blend"
print(f"submit: submission_{chosen}.csv  ({elapsed_hours():.2f}h elapsed total)")